In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv("../Raw-Data/Ling.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (2859, 3)
Columns: ['subject', 'body', 'label']


,subject,body,label
0,job posting - apple-iss research center,content - length : 3386 apple-iss research cen...,0
1,NaN,"lang classification grimes , joseph e . and ba...",0
2,query : letter frequencies for text identifica...,i am posting this inquiry for sergei atamas ( ...,0
3,risk,a colleague and i are researching the differin...,0
4,request book information,earlier this morning i was on the phone with a...,0


In [4]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\r", " ").replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["subject_clean"] = df["subject"].apply(clean_text)
df["body_clean"] = df["body"].apply(clean_text)

df["email_text"] = (
    df["subject_clean"] + " " + df["body_clean"]
    ).str.strip()

df["label"] = pd.to_numeric(df["label"], errors="coerce")

In [6]:
print(df["label"].value_counts(dropna=False))
print(df[["subject", "body", "email_text", "label"]].head())

label
0    2401
1     458
Name: count, dtype: int64
                                             subject  \
0            job posting - apple-iss research center   
1                                                NaN   
2  query : letter frequencies for text identifica...   
3                                               risk   
4                           request book information   

                                                body  \
0  content - length : 3386 apple-iss research cen...   
1  lang classification grimes , joseph e . and ba...   
2  i am posting this inquiry for sergei atamas ( ...   
3  a colleague and i are researching the differin...   
4  earlier this morning i was on the phone with a...   

                                          email_text  label  
0  job posting - apple-iss research center conten...      0  
1  lang classification grimes , joseph e . and ba...      0  
2  query : letter frequencies for text identifica...      0  
3  risk a colleague and i 

In [7]:
clean_df = df[["email_text", "label"]].copy()

clean_df = clean_df.dropna(subset=["email_text", "label"])
clean_df = clean_df[clean_df["email_text"].str.len() > 0]
clean_df = clean_df.drop_duplicates(subset=["email_text"]).reset_index(drop=True)

clean_df["label"] = clean_df["label"].astype(int)

print("Cleaned shape:", clean_df.shape)
print(clean_df["label"].value_counts())
clean_df.head()

Cleaned shape: (2859, 2)
label
0    2401
1     458
Name: count, dtype: int64


,email_text,label
0,job posting - apple-iss research center conten...,0
1,"lang classification grimes , joseph e . and ba...",0
2,query : letter frequencies for text identifica...,0
3,risk a colleague and i are researching the dif...,0
4,request book information earlier this morning ...,0


In [8]:
clean_df["email_text"] = clean_df["email_text"].astype(str)

clean_df["email_text"].str.contains(
    r"\b(?:spam|ham|phishing|legitimate)\b",
    case=False,
    na=False
).mean()

np.float64(0.02378454004896817)

In [9]:
clean_df.to_csv("../Processed-Data/ling_cleaned.csv", index=False)
print("Ling cleaned dataset saved.")

Ling cleaned dataset saved.
